# Benchmark de transcripción en GPU — Tarea 5

Este notebook corre en **Google Colab con GPU** y mide lo que pide la fase 2 de la Tarea 1:
dos tamaños de modelo, GPU contra CPU, float16 contra int8, glosario y **VRAM medida**.

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.
2. Sube tus 10 audios (`E01.m4a` … `V05.m4a`) a una carpeta de Google Drive llamada **`Tarea5_audio`**.
3. `Entorno de ejecución → Ejecutar todo`. Tarda unos 20–30 minutos.

In [ ]:
# 1. Evidencia de hardware: qué GPU asignó Colab
!nvidia-smi

In [ ]:
# 2. Código del proyecto (repositorio público)
REPO = "https://github.com/henryluis0603/Tarea-5-Data-Science-con-Python.git"
RAMA = "feat/evaluacion-gpu"   # cuando los PRs estén fusionados, también sirve "main"
!rm -rf /content/proyecto && git clone -q -b $RAMA $REPO /content/proyecto
%cd /content/proyecto
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. Audios desde Google Drive (no están en el repositorio)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p audio && cp /content/drive/MyDrive/Tarea5_audio/*.m4a audio/ && ls -la audio

In [ ]:
# 4. El dispositivo se lee de la librería en tiempo de ejecución, no de config.yaml
import ctranslate2, faster_whisper
print("faster-whisper", faster_whisper.__version__, "| ctranslate2", ctranslate2.__version__)
print("GPUs CUDA visibles para CTranslate2:", ctranslate2.get_cuda_device_count())

In [ ]:
# 5. Benchmark completo (modelos x precisiones x dispositivo + glosario)
!python -m scripts.benchmark_gpu --clips-cpu E01 E02

In [ ]:
# 6. Resumen
import json, pandas as pd
r = json.load(open("results/benchmark_gpu.json"))
df = pd.DataFrame(r["filas"]).drop(columns=["texto"])
print("GPU:", r["gpu"])
display(df.groupby(["modelo","dispositivo_pedido","dispositivo_real","precision","modo_glosario"])
          [["wer","cer","seg_por_min_audio","vram_pico_mb","vram_modelo_mb"]].mean().round(3))
print("¿Cambia el texto float16 vs int8?"); display(r["precision_cambia_texto"])
print("¿Cambia el texto GPU vs CPU?"); display(r["cpu_vs_gpu_cambia_texto"])

In [ ]:
# 7. Descarga los resultados y ponlos en la carpeta del proyecto en tu computadora
!zip -qr resultados_gpu.zip results/benchmark_gpu.json data/transcripciones/benchmark
from google.colab import files
files.download("resultados_gpu.zip")